# Exploratory Data Analysis of Montra Hospitality Intel

#### Import all Libraries

In [69]:
# Importing libraries
import os 
import pandas as pd
import plotly
from sqlalchemy import create_engine, text
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.express as px
import html
import re
import plotly.graph_objects as go
import plotly.subplots as sub
from plotly.subplots import make_subplots as msub 
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from turtle import title

# ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# RAW, PROC, OUT = ROOT / "data" / "raw", ROOT / "data" / "processed", ROOT / "outputs"
# for p in (PROC, OUT): p.mkdir(parents=True, exist_ok=True)
SEED = 42
LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#D1495B", "neutral": "#B8BCC4", "positive": "#2E8B57"}
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

#### Load Cleaned Montra Dataset

In [48]:
# Load cleaned data file
cleaned_data = pd.read_csv(
    r"C:\Users\User\Documents\AMDARI_IMS\Montra_Hospitality_Intel_PJ\Dataset\cleaned_data\cleaned_montra_data.csv")
 
 # converting the datetime
date_cols = ["review_date", "check_in_date", "check_out_date"]

for col in date_cols:
    cleaned_data[col] = pd.to_datetime(cleaned_data[col], errors="coerce")

# Data Summary
data_summary = pd.DataFrame({
    "Data Type": cleaned_data.dtypes.astype(str),
    "Missing Values": cleaned_data.isna().sum(),
    "% Missing Values": (cleaned_data.isna().mean() * 100).round(2),
    "Unique Values": cleaned_data.nunique(),
    "Duplicate Values": [cleaned_data.duplicated(subset=[col]).sum() for col in cleaned_data.columns]
}).sort_values(by="% Missing Values", ascending=False)

data_summary

,Data Type,Missing Values,% Missing Values,Unique Values,Duplicate Values
loyalty_status,object,17347,50.34,3,34456
review_id,int64,0,0.00,34460,0
check_out_date,datetime64[ns],0,0.00,531,33929
total_rooms,int64,0,0.00,23,34437
star_rating,int64,0,0.00,3,34457
location,object,0,0.00,14,34446
hotel_name,object,0,0.00,24,34436
previous_stays,int64,0,0.00,13,34447
country,object,0,0.00,15,34445
customer_type,object,0,0.00,5,34455


#### Bivariete EDA


In [ ]:
####### 1. ----------Booking Analysis Overview----------
#=================================================

# Total Booked Room_type
rm_type = cleaned_data['room_type'].value_counts().reset_index()
print(f"Total Rooms Booked:\n{rm_type}")
print("-"* 30)


###------ Revenue generated by room type
rm_revenue = cleaned_data.groupby('room_type')['total_amount'].sum().reset_index()
print(f" Total Rvenue Generated: \n{rm_revenue}")
print("-"* 35)

###------- Booking Channel Distribution
channels = cleaned_data['booking_channel'].value_counts().reset_index()
print(f"Total Booking Channel Counts: \n{channels}")
print("-"* 30)

###------- Revenue generated by booking channels
bc_revenue = cleaned_data.groupby('booking_channel')['total_amount'].sum().reset_index()
print(f"Book Channel Revenue: \n{bc_revenue}")
print("-"* 30)

###------ Check for guest stay duration
stay_duration = (
    cleaned_data["stay_duration"].value_counts().rename_axis("stay_duration")
    .reset_index(name="count").sort_values("stay_duration"))

print(f"Guest Stay Duration:\n {stay_duration}")


Total Rooms Booked:
     room_type  count
0     Standard  10440
1       Deluxe   8503
2    Executive   5199
3  Family Room   4076
4        Suite   3485
5         Twin   2757
------------------------------
 Total Rvenue Generated: 
     room_type  total_amount
0       Deluxe    3888702.19
1    Executive    3153725.03
2  Family Room    2121120.84
3     Standard    3067529.66
4        Suite    2970988.93
5         Twin     844378.53
-----------------------------------
Total Booking Channel Counts: 
    booking_channel  count
0    Direct Website   8537
1       Booking.com   7683
2           Expedia   5198
3      Travel Agent   3474
4            Airbnb   3372
5  Corporate Portal   3355
6           Walk-in   2841
------------------------------
Book Channel Revenue: 
    booking_channel  total_amount
0            Airbnb    1578579.42
1       Booking.com    3590018.82
2  Corporate Portal    1580017.69
3    Direct Website    3977792.32
4           Expedia    2419570.08
5      Travel Agent    15

In [93]:
####----------Visualization of Bookings
fig = msub(rows =3,cols= 2, 
           subplot_titles= ("Total Room Distribution", "Revenue Generated by Room Type", "Guest Booking Channels",
                            "Revenue Generated by Booking Channels", "Stay Duration Distribution (Days)", ),
           specs=[[{}, {}], [{}, {}], [{"colspan": 2}, None]])

## Add Charts
fig.add_trace(go.Bar(x= rm_type['room_type'], y = rm_type['count'], name = 'Counts', marker_color = 'teal'), row = 1, col = 1)
fig.add_trace(go.Bar(x= rm_revenue['room_type'], y = rm_revenue['total_amount'], name = 'Revenue', marker_color ='indianred'), row = 1, col = 2)
fig.add_trace(go.Bar(x= channels['booking_channel'], y = channels['count'], name = 'Counts', marker_color = 'blue'), row = 2, col = 1)
fig.add_trace(go.Bar(x= bc_revenue['booking_channel'], y = bc_revenue['total_amount'], name = 'Reveneue', marker_color = 'gold'), row =2, col =2)
fig.add_trace(go.Bar(x= stay_duration['stay_duration'], y =stay_duration['count'], name = 'Days Booked', marker_color = 'goldenrod'), row =3, col = 1)

fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  height = 800, title_text = 'Booking Analysis Overview', showlegend = False)


In [71]:
cleaned_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34460 entries, 0 to 34459
Data columns (total 25 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   review_id              34460 non-null  int64         
 1   booking_id             34460 non-null  int64         
 2   guest_id               34460 non-null  int64         
 3   hotel_id               34460 non-null  int64         
 4   review_text            34460 non-null  object        
 5   rating_score           34460 non-null  float64       
 6   review_date            34460 non-null  datetime64[ns]
 7   sentiment_label        34460 non-null  object        
 8   guest_id_booking_data  34460 non-null  int64         
 9   hotel_id_booking_data  34460 non-null  int64         
 10  room_type              34460 non-null  object        
 11  booking_channel        34460 non-null  object        
 12  check_in_date          34460 non-null  datetime64[ns]
 13  c

In [89]:
####### 2. ----------Guest Analysis Overview----------
#=================================================

# Total Guest Class
cust_type = cleaned_data['customer_type'].value_counts().reset_index()
print(f"Total Guest Class:\n{cust_type}")
print("-"* 30)

# Most Patronised Guest Status
ly_status = cleaned_data['loyalty_status'].value_counts().reset_index()
print(f"Total Loyal Guest:\n{ly_status}")
print("-"* 30)


###------ Revenue generated by country
cn_revenue = cleaned_data.groupby('country')['total_amount'].sum().reset_index()
print(f" Total Revenue Generated: \n{cn_revenue}")
print("-"* 35)


###------- most loyal guest class
# cust_status = cleaned_data.groupby('customer_type')['loyalty_status'].value_counts().reset_index()
# print(f"Guest loyalty Status: \n{cust_status}")
# print("-"* 30)

cust_status = (
    cleaned_data
    .groupby(['customer_type', 'loyalty_status'])
    .size()
    .reset_index(name='count')
)


Total Guest Class:
  customer_type  count
0       Leisure  13801
1      Business   8280
2     Corporate   5433
3         Group   3505
4        Family   3441
------------------------------
Total Loyal Guest:
  loyalty_status  count
0         Silver   8485
1           Gold   5754
2       Platinum   2874
------------------------------
 Total Revenue Generated: 
                 country  total_amount
0              Australia      78169.00
1                 Canada     579618.76
2                  China      90751.15
3                 France     929058.55
4                Germany    1173363.12
5                  India     163430.48
6                Ireland     828308.02
7                  Italy     484194.63
8            Netherlands     775060.95
9                Nigeria    1577396.26
10          South Africa     336926.97
11                 Spain     876071.93
12  United Arab Emirates     622646.27
13        United Kingdom    5570486.80
14         United States    1960962.29
---------------

In [96]:
####----------Visualization of Guest
fig = msub(rows =2,cols= 2, 
           subplot_titles= ("Guest Class Distribution", "Guest Loyalty status", "Revenue Generated by Country",
                            "Guest loyalty Status"),
           specs=[[{}, {}], [{}, {}]])

## Add Charts
fig.add_trace(go.Bar(x= cust_type['customer_type'], y = cust_type['count'], name = 'Counts', marker_color = 'teal'), row = 1, col = 1)
fig.add_trace(go.Bar(x= ly_status['loyalty_status'], y = ly_status['count'], name = 'Counts', marker_color ='indianred'), row = 1, col = 2)
fig.add_trace(go.Bar(x= cn_revenue['country'], y = cn_revenue['total_amount'], name = 'Revenue', marker_color = 'blue'), row = 2, col = 1)
# Row 2, col 2: one Bar trace per loyalty status, stacked
colors = ['gold', 'blue', 'red', 'green', 'gray']
for color, (status, grp) in zip(colors, cust_status.groupby('loyalty_status')):
    fig.add_trace(go.Bar(x=grp['customer_type'], y=grp['count'], name=status, marker_color=color),
        row=2, col=2)

fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  height = 800, title_text = 'Guest Analysis Overview', barmode= 'stack', showlegend = False)


In [103]:
####### 3. ----------Hotel Analysis Overview----------
#=================================================

# Total Booked Hotels
h_count = cleaned_data['hotel_name'].value_counts().reset_index()
print(f"Total Booked Hotels:\n{h_count}")
print("-"* 30)


###------ Revenue generated by by Hotels
h_revenue = cleaned_data.groupby('hotel_name')['total_amount'].sum().reset_index()
print(f" Total Revenue Generated: \n{h_revenue}")
print("-"* 35)


###------- Total Hotel Roooms
h_rooms = cleaned_data.groupby('hotel_name')['total_rooms'].value_counts().reset_index()
print(f"Book Channel Revenue: \n{h_rooms}")
print("-"* 30)

###------ Hotel Ratings
h_star_ratings = (
    cleaned_data.groupby("hotel_name")['star_rating'].value_counts().reset_index(name="count").sort_values("star_rating"))

print(f"Hotel Rating:\n {stay_duration}")


Total Booked Hotels:
                    hotel_name  count
0    Edinburgh Riverside Hotel   2865
1       Southampton Park Hotel   1438
2   Nottingham Riverside Hotel   1423
3        Glasgow Skyline Hotel   1412
4         The Belfast Heritage   1411
5        Nottingham Park Hotel   1410
6         Glasgow Metropolitan   1407
7      Manchester Metropolitan   1404
8          The London Boutique   1394
9         The Belfast Boutique   1384
10     The Birmingham Heritage   1384
11       The Brighton Heritage   1383
12     Liverpool Harbour Hotel   1379
13         The Oxford Boutique   1374
14        The Birmingham Grand   1370
15         The London Heritage   1369
16       Glasgow Harbour Hotel   1363
17             The Leeds Grand   1350
18        Belfast Metropolitan   1350
19        The Glasgow Heritage   1337
20          The Aberdeen Grand   1336
21     Leicester Harbour Hotel   1330
22         London Metropolitan   1313
23      Liverpool Metropolitan   1274
-----------------------------

In [110]:
####----- Visualzation of the Best Hotels
TOP_N = 10

# Top 10 for each chart
top_count   = h_count.nlargest(TOP_N, 'count')
top_revenue = h_revenue.nlargest(TOP_N, 'total_amount')
top_rooms   = h_rooms.nlargest(TOP_N, 'total_rooms')

# Top 10 hotels by total count, then keep only their star-rating rows
top_hotels = (
    h_star_ratings.groupby('hotel_name')['count'].sum().nlargest(TOP_N).index)
top_ratings = h_star_ratings[h_star_ratings['hotel_name'].isin(top_hotels)]

fig = msub(rows=2, cols=2,subplot_titles=("10 Most Booked Hotels", "Top 10 Hotels by Revenue",
                    "Top 10 Hotels by Rooms", "Best 10 Most Rated Hotels"),specs=[[{}, {}], [{}, {}]])

# Add charts
fig.add_trace(go.Bar(x=top_count['hotel_name'], y=top_count['count'],
                     name='Counts', marker_color='teal', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(x=top_revenue['hotel_name'], y=top_revenue['total_amount'],
                     name='Revenue', marker_color='indianred', showlegend=False), row=1, col=2)

fig.add_trace(go.Bar(x=top_rooms['hotel_name'], y=top_rooms['total_rooms'],
                     name='Hotel Rooms', marker_color='blue', showlegend=False), row=2, col=1)

# Top 10 Best Hotels
colors = ['gold', 'blue', 'red', 'green', 'gray']
for color, (star, grp) in zip(colors, top_ratings.groupby('star_rating')):
    fig.add_trace(go.Bar(x=grp['hotel_name'], y=grp['count'],
                         name=f"{star} star", marker_color=color), row=2, col=2)

fig.update_xaxes(tickangle=-45)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
    height=800, title_text='Hotel Analysis Overview', barmode='stack', showlegend=True,)

In [111]:
####### 4. ----------Hotel Review Analysis Overview----------
#=================================================

# Total Booked Room_type
rm_type = cleaned_data['room_type'].value_counts().reset_index()
print(f"Total Rooms Booked:\n{rm_type}")
print("-"* 30)


###------ Revenue generated by room type
rm_revenue = cleaned_data.groupby('room_type')['total_amount'].sum().reset_index()
print(f" Total Rvenue Generated: \n{rm_revenue}")
print("-"* 35)

###------- Booking Channel Distribution
channels = cleaned_data['booking_channel'].value_counts().reset_index()
print(f"Total Booking Channel Counts: \n{channels}")
print("-"* 30)

###------- Revenue generated by booking channels
s_rating = cleaned_data.groupby('sentiment_label')['rating_score'].value_counts().reset_index()
print(f"Hotel Rating by Sentiments: \n{s_rating}")
print("-"* 30)

###------ Check for guest stay duration
stay_duration = (
    cleaned_data["stay_duration"].value_counts().rename_axis("stay_duration")
    .reset_index(name="count").sort_values("stay_duration"))

print(f"Guest Stay Duration:\n {stay_duration}")


Total Rooms Booked:
     room_type  count
0     Standard  10440
1       Deluxe   8503
2    Executive   5199
3  Family Room   4076
4        Suite   3485
5         Twin   2757
------------------------------
 Total Rvenue Generated: 
     room_type  total_amount
0       Deluxe    3888702.19
1    Executive    3153725.03
2  Family Room    2121120.84
3     Standard    3067529.66
4        Suite    2970988.93
5         Twin     844378.53
-----------------------------------
Total Booking Channel Counts: 
    booking_channel  count
0    Direct Website   8537
1       Booking.com   7683
2           Expedia   5198
3      Travel Agent   3474
4            Airbnb   3372
5  Corporate Portal   3355
6           Walk-in   2841
------------------------------
Hotel Rating by Sentiments: 
   sentiment_label  rating_score  count
0         negative           2.4   1133
1         negative           2.3   1008
2         negative           2.2    953
3         negative           2.1    922
4         negative     

In [ ]:
####----------Visualization of reviews
fig = msub(rows =2,cols= 2, 
           subplot_titles= ("Guest Class Distribution", "Guest Loyalty status", "Revenue Generated by Country",
                            "Guest loyalty Status"),
           specs=[[{}, {}], [{}, {}]])

## Add Charts
fig.add_trace(go.Bar(x= cust_type['customer_type'], y = cust_type['count'], name = 'Counts', marker_color = 'teal'), row = 1, col = 1)
fig.add_trace(go.Bar(x= ly_status['loyalty_status'], y = ly_status['count'], name = 'Counts', marker_color ='indianred'), row = 1, col = 2)
fig.add_trace(go.Bar(x= cn_revenue['country'], y = cn_revenue['total_amount'], name = 'Revenue', marker_color = 'blue'), row = 2, col = 1)
# Row 2, col 2: one Bar trace per loyalty status, stacked
colors = ['gold', 'blue', 'red', 'green', 'gray']
for color, (status, grp) in zip(colors, cust_status.groupby('loyalty_status')):
    fig.add_trace(go.Bar(x=grp['customer_type'], y=grp['count'], name=status, marker_color=color),
        row=2, col=2)

fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  height = 800, title_text = 'Guest Analysis Overview', barmode= 'stack', showlegend = False)


In [112]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# review_date must be datetime for resampling
cleaned_data["review_date"] = pd.to_datetime(cleaned_data["review_date"], errors="coerce")

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        "Rating Distribution by Sentiment",
        "Sentiment Label Distribution",
        "Review Volume by Month",
        "Reviews per Hotel",
        "Average Rating per Hotel",
        "Review Length in Words",
    ),
)

# 1. Rating distribution by sentiment label
for label in LABELS:
    ratings = cleaned_data.loc[cleaned_data["sentiment_label"] == label, "rating_score"]
    fig.add_trace(
        go.Histogram(x=ratings, nbinsx=40, name=label,
                     marker_color=COLORS[label], opacity=0.85),
        row=1, col=1,
    )
fig.update_xaxes(title_text="Rating Score", row=1, col=1)

# 2. Sentiment label distribution
label_counts = cleaned_data["sentiment_label"].value_counts().reindex(LABELS)
fig.add_trace(
    go.Bar(
        x=label_counts.index,
        y=label_counts.values,
        marker_color=[COLORS[l] for l in label_counts.index],
        text=[f"{c:,}" for c in label_counts.values],
        textposition="outside",
        showlegend=False,
    ),
    row=1, col=2,
)

# 3. Number of reviews per month
monthly_reviews = (
    cleaned_data.dropna(subset=["review_date"])
    .set_index("review_date")
    .resample("MS")
    .size()
)
fig.add_trace(
    go.Scatter(x=monthly_reviews.index, y=monthly_reviews.values,
               mode="lines+markers", line=dict(color="#4C78A8"), showlegend=False),
    row=1, col=3,
)
fig.update_xaxes(tickangle=-45, row=1, col=3)

# 4. Number of reviews per hotel
hotel_reviews = cleaned_data.groupby("hotel_name").size().sort_values()
fig.add_trace(
    go.Bar(x=hotel_reviews.values, y=hotel_reviews.index, orientation="h",
           marker_color="#4C78A8", showlegend=False),
    row=2, col=1,
)
fig.update_yaxes(tickfont_size=7, row=2, col=1)

# 5. Average rating per hotel
hotel_ratings = (
    cleaned_data.groupby("hotel_name")["rating_score"].mean().sort_values()
)
fig.add_trace(
    go.Bar(x=hotel_ratings.values, y=hotel_ratings.index, orientation="h",
           marker_color="#F58518", showlegend=False),
    row=2, col=2,
)
fig.update_yaxes(tickfont_size=7, row=2, col=2)
# Keep the rating axis close to the observed values
fig.update_xaxes(range=[hotel_ratings.min() - 0.1, hotel_ratings.max() + 0.1], row=2, col=2)

# 6. Review length in words
review_lengths = cleaned_data["review_text"].str.split().str.len()
fig.add_trace(
    go.Histogram(x=review_lengths, nbinsx=40, marker_color="#72B7B2", showlegend=False),
    row=2, col=3,
)
fig.update_xaxes(title_text="Number of Words", row=2, col=3)

fig.update_layout(
    height=800, width=1400,
    barmode="overlay",       # overlaps the sentiment histograms, like alpha in matplotlib
    showlegend=True,         # legend only shows the sentiment labels
    plot_bgcolor="white", paper_bgcolor="white",
    title_text="Review Analysis Overview",
)
fig.show()